# LACT / simtelarray 事件检查

这个 notebook 按物理流程检查一个事件：读取事件、准备 raw image、图像清理、Hillas 参数、方向/芯位重建、SDP 平面图。

不同输入格式只在第一个配置 cell 里切换；读出来以后，后面的分析和画图步骤保持一致。

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(".") if Path("configs").exists() else Path("..")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".mplcache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

LACT_ROOT_FILE = PROJECT_ROOT / "run_logs/lactroot_only/lact_events.root"

SIMTEL_FILE = PROJECT_ROOT / "path/to/input.zst"
ROOT_EVENT_FILE = PROJECT_ROOT / "path/to/pylast_events.root"


# 只改这里即可："lact_root" / "simtel" / "root_event"
INPUT_KIND = "lact_root"
INPUT_FILE = LACT_ROOT_FILE
# INPUT_KIND = "simtel"
# INPUT_FILE = SIMTEL_FILE
# INPUT_KIND = "root_event"; INPUT_FILE = ROOT_EVENT_FILE

EVENT_INDEX = 0
MAX_EVENTS = 15

print("input kind:", INPUT_KIND)
print("input file:", INPUT_FILE)
print("exists:", INPUT_FILE.exists())


In [ ]:
import time
import numpy as np
from IPython.display import display

try:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"
except Exception:
    pio = None

from pylast.io import LactEventSource, SimtelEventSource, RootEventSource
from pylast.calib import Calibrator
from pylast.image import ImageProcessor
from pylast.reco import ShowerProcessor
from pylast.visualize import (
    EventVisualizer,
    hillas_parameter_rows,
    plot_clean_images,
    plot_event_cameras,
    plot_gathered_images,
    plot_event_cores,
    plot_event_trigger_timing,
    plot_event_sdp_planes,
    plot_event_sdp_planes_3d,
    plot_event_sdp_planes_3d_interactive,
    plot_raw_images,
    reconstruction_summary,
)


CALIBRATOR_CONFIG = """{
  "Calibrator": {
    "image_extractor_type": "LocalPeakExtractor",
    "LocalPeakExtractor": {
      "window_width": 7,
      "window_shift": 3,
      "apply_correction": false
    }
  }
}
"""

IMAGE_PROCESSOR_CONFIG = """{
  "poisson_noise": 0.0
}
"""

SHOWER_PROCESS_CONFIG = """{
  "ShowerProcessor": {
    "GeometryReconstructionTypes": ["HillasReconstructor"],
    "HillasReconstructor": {
      "use_fake_hillas": false,
      "ImageQuery": "hillas_intensity > 100 && leakage_intensity_width_2 < 0.3"
    }
  }
}
"""


def step(name, fn):
    print(f"BEGIN {name}", flush=True)
    t0 = time.perf_counter()
    out = fn()
    print(f"END {name}: {time.perf_counter() - t0:.3f}s", flush=True)
    return out


def read_one_event(kind, filename, event_index=0, max_events=10):
    filename = str(filename)
    if kind == "lact_root":
        source_data = LactEventSource(filename, max_events=max_events)
        event = source_data[event_index]
    elif kind == "simtel":
        source_data = SimtelEventSource(filename, max_events=max(max_events, event_index + 1))
        event = list(source_data)[event_index]
    elif kind == "root_event":
        source_data = RootEventSource(filename, max_events=max_events)
        event = source_data[event_index]
    else:
        raise ValueError("INPUT_KIND must be lact_root, simtel, or root_event")
    return source_data, event, EventVisualizer(source_data)


def build_processors(source_data):
    calibrator = Calibrator(source_data.subarray, config_str=CALIBRATOR_CONFIG)
    image_processor = ImageProcessor(source_data.subarray, config_str=IMAGE_PROCESSOR_CONFIG)
    shower_processor = ShowerProcessor(source_data.subarray, config_str=SHOWER_PROCESS_CONFIG)
    return calibrator, image_processor, shower_processor


def prepare_raw_image(event, calibrator):
    if getattr(event, "dl0", None) is None:
        step("抽取 raw image", lambda: calibrator(event))
    return event


def print_event_summary(event):
    shower = event.simulation.shower
    print("event_id:", event.event_id)
    print("run_id:", event.run_id)
    print("energy [TeV]:", float(shower.energy))
    print("true zenith [deg]:", 90.0 - np.degrees(float(shower.alt)))
    print("true azimuth [deg]:", np.degrees(float(shower.az)))
    print("true core [m]:", float(shower.core_x), float(shower.core_y))


## 1. 读取事件

这里读入一种格式。后面的 cell 不再区分输入来自 LACT ROOT、simtelarray，还是 pylast 原生 ROOT。

In [ ]:
source_data, event, visualizer = step(
    "读取事件",
    lambda: read_one_event(INPUT_KIND, INPUT_FILE, EVENT_INDEX, MAX_EVENTS),
)

print("subarray", len(source_data.subarray.tels), flush=True)

print("calibrator", flush=True)
calibrator = Calibrator(source_data.subarray, config_str=CALIBRATOR_CONFIG)

print("image_processor", flush=True)
image_processor = ImageProcessor(source_data.subarray, config_str=IMAGE_PROCESSOR_CONFIG)

print("shower_processor", flush=True)
shower_processor = ShowerProcessor(source_data.subarray, config_str=SHOWER_PROCESS_CONFIG)

print("done", flush=True)

In [ ]:
# 检查 LACT ROOT 里是否写入并能读出 ground 粒子数。
if hasattr(source_data, "get_ground_counts"):
    print("source.get_ground_counts(event):", source_data.get_ground_counts(event))
else:
    print("当前 source 没有 get_ground_counts 接口；这通常表示不是 LACT ROOT 输入。")

try:
    import ROOT
    root_path = getattr(source_data, "_input_filename", None)
    if root_path is None:
        print("没有 ROOT 输入文件路径，跳过 ROOT branch 检查。")
    else:
        f = ROOT.TFile.Open(str(root_path))
        t = f.Get("corsika_events") if f else None
        names = ["ground_gammas", "ground_electrons", "ground_hadrons", "ground_muons"]
        print("corsika_events exists:", bool(t))
        print("ground branches:", {name: bool(t and t.GetBranch(name)) for name in names})
        if t and t.GetEntries() > 0:
            target_event_id = int(getattr(event, "event_id", -1))
            for i in range(t.GetEntries()):
                t.GetEntry(i)
                if int(t.event_id) == target_event_id:
                    print("corsika_events row:", {name: getattr(t, name) for name in names})
                    break
        if f:
            f.Close()
except Exception as exc:
    print("ROOT branch 检查失败:", repr(exc))


In [ ]:
print("after processors", flush=True)
print("has dl0:", getattr(event, "dl0", None) is not None, flush=True)
print("has r1:", getattr(event, "r1", None) is not None, flush=True)
print("dl0 tels:", len(event.dl0.tels) if getattr(event, "dl0", None) else None, flush=True)
print("r1 tels:", len(event.r1.tels) if getattr(event, "r1", None) else None, flush=True)

print("before prepare_raw_image", flush=True)
prepare_raw_image(event, calibrator)
print("after prepare_raw_image", flush=True)

print("before summary", flush=True)
print_event_summary(event)
print("after summary", flush=True)

In [ ]:
# source_data, event, visualizer = step(
#     "读取事件",
#     lambda: read_one_event(INPUT_KIND, INPUT_FILE, EVENT_INDEX, MAX_EVENTS),
# )
# calibrator, image_processor, shower_processor = build_processors(source_data)
# prepare_raw_image(event, calibrator)
# print_event_summary(event)


## 2. 望远镜分布、触发时延、芯位和 SDP 投影

这里画阵列背景、真实芯位、触发望远镜、相对触发时延与 p.e.，以及 SDP 平面在地面的投影。

In [ ]:
plot_event_cores(event, visualizer=visualizer, include_non_triggered=False)


In [ ]:
# 颜色表示相对最早触发望远镜的时间延迟，点面积表示总 p.e.。
# 阵列背景、真实 core 和入射方向沿用 plot_event_cores 的画法。
trigger_timing = (
    source_data.get_trigger_timing(event)
    if hasattr(source_data, "get_trigger_timing") else {}
)
if trigger_timing:
    plot_event_trigger_timing(
        event,
        visualizer=visualizer,
        image_level="dl0",
        annotate=True,
    )
else:
    print("当前输入没有望远镜触发时间；请使用新版 LACT ROOT 输出。")


In [ ]:
plot_event_sdp_planes(event, visualizer=visualizer, include_non_triggered=False)


In [ ]:
plot_event_cameras(
    event,
    visualizer=visualizer,
    image_level="simulation",
    include_non_triggered=False,
)

## 3. Raw image

这里画清理前的相机图像。对 LACT ROOT，这是输出里的积分 p.e.；对 simtelarray，这是从 waveform 抽取出的积分图像。

In [ ]:
plot_raw_images(event, visualizer=visualizer, include_non_triggered=False)

In [ ]:
plot_gathered_images(
    event,
    visualizer=visualizer,
    image_type="raw",
    show_hillas=False,
    include_non_triggered=False,
)


## 4. Clean image 和 Hillas 参数

这里运行图像清理，并画清理后的图像和 Hillas 椭圆。

In [ ]:
step("图像清理 + Hillas 参数", lambda: image_processor(event))

rows = hillas_parameter_rows(event)
print("有 Hillas 参数的望远镜:", [row["tel_id"] for row in rows])
for row in rows:
    print(
        f"Tel {row['tel_id']:2d}: intensity={row['intensity']:.2f}, "
        f"length={row['length_rad']:.5g} rad, width={row['width_rad']:.5g} rad, "
        f"psi={np.degrees(row['psi_rad']):.2f} deg"
    )

# plot_clean_images(event, visualizer=visualizer, show_hillas=True, include_non_triggered=False,ideal=True)



In [ ]:
plot_clean_images(
    event,
    visualizer=visualizer,
    show_hillas=True,
    include_non_triggered=False,
    ideal=True,
)

## 5. 方向和芯位重建

这里用 pylast 的 Hillas 重建。若 `is_valid=False`，通常说明通过筛选的望远镜不够或图像质量条件太严。

In [ ]:
step("方向/芯位重建", lambda: shower_processor(event))
summary = reconstruction_summary(event, "HillasReconstructor")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6g}")
    else:
        print(f"{key}: {value}")


In [ ]:
plot_clean_images(
    event,
    visualizer=visualizer,
    show_hillas=True,
    include_non_triggered=False,
    ideal=True,
    reco=True,
)

In [ ]:
plot_gathered_images(
    event,
    visualizer=visualizer,
    image_type="clean",
    show_hillas=True,
    include_non_triggered=False,
    ideal=True,
    reco=True,
)


## 6. 3D SDP 平面

这里在重建之后画 3D SDP。默认先画 Plotly 交互图，可以在 notebook 或导出的 HTML 里旋转、缩放；红色是真实芯位和真实方向，蓝色是重建芯位和重建方向。

In [ ]:
plot_event_sdp_planes_3d_interactive(
    event,
    visualizer=visualizer,
    include_non_triggered=False,
    z_max=1200.0,
    show_reco=True,
)


In [ ]:
# 静态版本用于保存 PNG 或快速预览。
plot_event_sdp_planes_3d(
    event,
    visualizer=visualizer,
    include_non_triggered=False,
    z_max=1200.0,
    show_reco=True,
    show=True,
)


## 7. 保存图片

可选。保存的文件名使用物理含义：core、sdp、raw、clean、sdp_3d。交互 3D 额外保存为 HTML。

In [ ]:
# OUTPUT_DIR = Path(INPUT_FILE).parent / "pylast_visualize"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# plots = {
#     "core": core_result,
#     "sdp": sdp_result,
#     "raw": raw_result,
#     "clean": clean_result,
#     "sdp_3d": sdp_3d_result,
# }
# for name, result in plots.items():
#     fig = result.get("figure")
#     if fig is None:
#         continue
#     path = OUTPUT_DIR / f"event_{event.event_id}_{name}.png"
#     fig.savefig(path, dpi=200, bbox_inches="tight")
#     print(path)

# html_path = OUTPUT_DIR / f"event_{event.event_id}_sdp_3d_interactive.html"
# sdp_3d_interactive_figure.write_html(html_path, include_plotlyjs="cdn", full_html=True)
# print(html_path)


## 项目与分支

- LACT_sim：<https://github.com/Yun532/LACT_sim>，分支 `user_v2.0`
- pyLAST：<https://github.com/Yun532/pylast>，分支 `lact_sim`

从零拉取：

```bash
git clone --branch user_v2.0 --single-branch https://github.com/Yun532/LACT_sim.git
git clone --branch lact_sim --single-branch https://github.com/Yun532/pylast.git
cd pylast
python -m pip install -e . --no-build-isolation
```

已有仓库时更新对应分支：

```bash
cd LACT_sim
git switch user_v2.0
git pull --ff-only origin user_v2.0
cd ../pylast
git switch lact_sim
git pull --ff-only origin lact_sim
python -m pip install -e . --no-build-isolation
```

本 notebook 在 LACT_sim 仓库中的相对路径为 `notebooks/lact_root_to_pylast_visualize.ipynb`。
